In [13]:
import geopandas as gpd
import pandas as pd

In [14]:
# Lista dos arquivos shapefiles em ordem decrescente de ano
shapefiles = [
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_id_2022.shp",
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_2020.shp",
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_2018.shp",
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_2014.shp",
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_2012.shp",
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_2010.shp",
    "~/Documents/VS_idade/shapes/corrigidos/VS_bacia_corrigido_2008.shp",
]

# Caminho para salvar os resultados
output_dir = "~/Documents/VS_idade/Results/"
final_file = f"{output_dir}VS_bacia_com_idade.shp"

In [15]:
# Função para corrigir geometrias inválidas automaticamente
def correct_invalid_geometries(layer):
    # Aplicar buffer(0) para corrigir geometrias inválidas
    layer['geometry'] = layer.geometry.buffer(0)
    return layer

In [16]:
def increment_character(column):
    digit = column[-1]
    value = int(digit)
    increment = value + 1
    return column[:-1] + str(increment)

In [17]:
def rename_columns(layer): 
    layer.columns = [increment_character(col) if '_' in col else col for col in layer.columns]
    return

In [18]:
# Loop para iterar entre os shapefiles
result_file = shapefiles[0]  # Primeiro arquivo (2022) é a base inicial
for i in range(1, len(shapefiles)):
    layer_A = gpd.read_file(result_file)  # Resultado anterior ou camada inicial
    layer_B = gpd.read_file(shapefiles[i])  # Próxima camada na sequência

    # Corrigir geometrias inválidas, se houver
    layer_A = correct_invalid_geometries(layer_A)
    layer_B = correct_invalid_geometries(layer_B)

    # Renomear colunas para evitar duplicação 
    rename_columns(layer_A)
    rename_columns(layer_B)

    # Verificar se as geometrias estão válidas
    if not (layer_A.is_valid.all() and layer_B.is_valid.all()):
        print(f"Geometrias inválidas corrigidas nos arquivos {result_file} ou {shapefiles[i]}")

    # Realizar o overlay (identity)
    result = gpd.overlay(layer_A, layer_B, how='identity')

    # Remover duplicações, se necessário
    result = result.drop_duplicates(subset='geometry')

    # Definir o nome do próximo arquivo de resultado
    year_B = shapefiles[i].split("_")[-1].replace(".shp", "")  # Extrair o ano de layer_B
    result_file = f"{output_dir}22_to_{year_B}.shp"

    # Salvar o resultado
    result.to_file(result_file)

    print(f"Overlay completo: {result_file}")

print("Processamento completo!")


/tmp/ipykernel_700059/4223858877.py:20: UserWarning: `keep_geom_type=True` in overlay resulted in 31 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = gpd.overlay(layer_A, layer_B, how='identity')


Overlay completo: ~/Documents/VS_idade/Results/22_to_2020.shp


/tmp/ipykernel_700059/4223858877.py:20: UserWarning: `keep_geom_type=True` in overlay resulted in 405 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = gpd.overlay(layer_A, layer_B, how='identity')


Overlay completo: ~/Documents/VS_idade/Results/22_to_2018.shp


/tmp/ipykernel_700059/4223858877.py:20: UserWarning: `keep_geom_type=True` in overlay resulted in 5598 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = gpd.overlay(layer_A, layer_B, how='identity')


...

Overlay completo: ~/Documents/VS_idade/Results/22_to_2012.shp


/tmp/ipykernel_700059/4223858877.py:20: UserWarning: `keep_geom_type=True` in overlay resulted in 7515 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = gpd.overlay(layer_A, layer_B, how='identity')


Overlay completo: ~/Documents/VS_idade/Results/22_to_2010.shp


/tmp/ipykernel_700059/4223858877.py:20: UserWarning: `keep_geom_type=True` in overlay resulted in 7787 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = gpd.overlay(layer_A, layer_B, how='identity')


Overlay completo: ~/Documents/VS_idade/Results/22_to_2008.shp
Processamento completo!


In [19]:
gdf = gpd.read_file(result_file)
refVal = gdf.iloc[0][1]
colunas = gdf.filter(like='ano').columns

/tmp/ipykernel_700059/1253848721.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  refVal = gdf.iloc[0][1]


In [20]:
def calcular_idade(row):
    if row[colunas[1:]].isnull().all():
        return 1
    # Percorrer as colunas em ordem
    for i in range(len(colunas) - 1):
        if pd.isnull(row[colunas[i + 1]]):
            return max(1, int(refVal - row[colunas[i]]))
    return int(refVal - row[colunas[-1]])

In [21]:

gdf['Idade'] = gdf.apply(calcular_idade, axis=1)
gdf.to_file(final_file)


Adicionei abaixo para calcular a idade ponderada

In [32]:
# Calcular a área de cada polígono em hectares
gdf['area_ha'] = round(gdf.geometry.area / 10_000, 2)  # Convertendo de m² para hectares, mantendo duas casas decimais


In [33]:
def calcular_idade_ponderada_por_id(gdf):
    # Agrupar por `Id` e calcular idade ponderada
    grouped = gdf.groupby('Id').apply(
        lambda group: pd.Series({
            'geometry': group.geometry.unary_union,  # Unir geometrias por `Id`
            'area_ha': group['area_ha'].sum(),  # Soma total da área
            'idade_ponderada': (group['area_ha'] * group['Idade']).sum() / group['area_ha'].sum()
        })
    ).reset_index()

    # Transformar a idade ponderada em inteiro
    grouped['idade_ponderada'] = grouped['idade_ponderada'].round().astype(int)
    return grouped


In [34]:
# Calcular idade ponderada
resultado_ponderado = calcular_idade_ponderada_por_id(gdf)

# Salvar o resultado final em um shapefile
resultado_ponderado = gpd.GeoDataFrame(resultado_ponderado, geometry='geometry', crs=gdf.crs)
resultado_ponderado.to_file(final_file)
print(f"Resultado final com idade ponderada salvo em: {final_file}")


/tmp/ipykernel_700059/1843888625.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  'geometry': group.geometry.unary_union,  # Unir geometrias por `Id`


Resultado final com idade ponderada salvo em: ~/Documents/VS_idade/Results/VS_bacia_com_idade.shp


/tmp/ipykernel_700059/1843888625.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = gdf.groupby('Id').apply(
/tmp/ipykernel_700059/1820626208.py:6: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  resultado_ponderado.to_file(final_file)
/home/jovyan/.local/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'idade_ponderada' to 'idade_pond'
  ogr_write(
